# Milestone 2: End-to-End Multi-Agent NDA Review Pipeline

Single Kaggle-runnable notebook that consolidates the MS2 deliverables:

1. **Vector RAG indexing** of the ContractNLI training split → ChromaDB at `/kaggle/working/vector_db`.
2. **Graph RAG indexing** of the same split → NetworkX pickle at `/kaggle/working/graph_db/contractnli_graph.pkl`.
3. **Retrievers** (`VectorRetriever`, `GraphRetriever`) that load those persisted indices.
4. **Stateful Conversational Agent** loading Qwen2.5-3B-Instruct in 4-bit with the MS1 LoRA adapter from `model-weights/final_adapter/` (PEFT).
5. **Orchestrator** that routes a request through exactly one retrieval branch, runs a multi-turn conversation, and emits a runtrace conforming to `assets/runtrace_ms2.schema.json`.

Switch retrieval mode by changing `RETRIEVER_MODE` in the configuration cell (`"vector"` or `"graph"`).

## 0. Configuration & Environment

In [1]:
import os

# Detect Kaggle vs local. On Kaggle, mount the dataset that contains contract-nli
# and add the model-weights folder as a Kaggle Dataset/Model input.
ON_KAGGLE = os.path.exists("/kaggle/working")

if ON_KAGGLE:
    WORK_DIR = "/kaggle/working"
    # Expected: a Kaggle dataset attached at /kaggle/input/contract-nli/ that holds train.json and test.json
    TRAIN_PATH = "/kaggle/input/datasets/yousefelbrolosy/contract-dataset/assets/train.json"
    EVAL_PATH = "/kaggle/input/datasets/yousefelbrolosy/contract-dataset/assets/test.json"
    # Expected: model-weights uploaded as a Kaggle Dataset, e.g. 'final-adapter-m2'
    ADAPTER_PATH_CANDIDATES = [
        "/kaggle/input/datasets/yousefelbrolosy/final-adapter-m2/final_adapter",
    ]
else:
    WORK_DIR = os.path.abspath(".")
    TRAIN_PATH = os.path.join(WORK_DIR, "sample.json")
    EVAL_PATH = os.path.join(WORK_DIR, "sample.json")
    ADAPTER_PATH_CANDIDATES = [os.path.join(WORK_DIR, "model-weights", "final_adapter")]

VECTOR_DB_DIR = os.path.join(WORK_DIR, "vector_db")
GRAPH_DB_DIR = os.path.join(WORK_DIR, "graph_db")
GRAPH_PKL = os.path.join(GRAPH_DB_DIR, "contractnli_graph.pkl")
RUNTRACE_OUT = os.path.join(WORK_DIR, "results_ms2_runtrace.json")

ADAPTER_PATH = next((p for p in ADAPTER_PATH_CANDIDATES if os.path.exists(p)), None)
BASE_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# Pipeline mode: route through exactly one retrieval branch per MS2 spec.
RETRIEVER_MODE = os.environ.get("RETRIEVER_MODE", "vector")  # "vector" or "graph"
assert RETRIEVER_MODE in {"vector", "graph"}, RETRIEVER_MODE

print(f"ON_KAGGLE         = {ON_KAGGLE}")
print(f"TRAIN_PATH        = {TRAIN_PATH}")
print(f"EVAL_PATH         = {EVAL_PATH}")
print(f"VECTOR_DB_DIR     = {VECTOR_DB_DIR}")
print(f"GRAPH_PKL         = {GRAPH_PKL}")
print(f"ADAPTER_PATH      = {ADAPTER_PATH}")
print(f"RETRIEVER_MODE    = {RETRIEVER_MODE}")

ON_KAGGLE         = True
TRAIN_PATH        = /kaggle/input/datasets/yousefelbrolosy/contract-dataset/assets/train.json
EVAL_PATH         = /kaggle/input/datasets/yousefelbrolosy/contract-dataset/assets/test.json
VECTOR_DB_DIR     = /kaggle/working/vector_db
GRAPH_PKL         = /kaggle/working/graph_db/contractnli_graph.pkl
ADAPTER_PATH      = /kaggle/input/datasets/yousefelbrolosy/final-adapter-m2/final_adapter
RETRIEVER_MODE    = vector


In [2]:
%pip install -q chromadb sentence-transformers networkx peft==0.18.1 transformers accelerate bitsandbytes jsonschema


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

## 1. Vector RAG Index (ContractNLI Training Split)

Structured chunks `Hypothesis + Outcome + Supporting Clause` extracted from labeled spans (no raw-text chunking).

In [3]:
import json
import chromadb
from sentence_transformers import SentenceTransformer

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_data = json.load(f)
print(f"Loaded {len(train_data.get('documents', []))} training documents")

labels_map = train_data.get("labels", {})

def extract_structured_chunks(data):
    chunks = []
    chunk_idx = 0
    for doc in data.get("documents", []):
        doc_id = str(doc["id"])
        text = doc["text"]
        doc_spans = doc.get("spans", [])
        annotations = doc.get("annotation_sets", [{}])[0].get("annotations", {})
        for hyp_id, ann in annotations.items():
            choice = ann.get("choice")
            span_indices = ann.get("spans", [])
            hypothesis_text = labels_map.get(hyp_id, {}).get("hypothesis", "")
            if choice in ("Entailment", "Contradiction") and span_indices:
                for idx in span_indices:
                    if idx >= len(doc_spans):
                        continue
                    span_val = doc_spans[idx]
                    if isinstance(span_val, list) and len(span_val) == 2:
                        ev_text = text[span_val[0]:span_val[1]]
                    elif isinstance(span_val, str):
                        ev_text = span_val
                    else:
                        continue
                    chunk_text = (
                        f"Hypothesis: '{hypothesis_text}'\n"
                        f"Historical Outcome: {choice}\n"
                        f"Supporting Clause: '{ev_text}'"
                    )
                    chunks.append({
                        "id": f"{doc_id}_{chunk_idx}",
                        "text": chunk_text,
                        "metadata": {"source_doc": doc_id, "hypothesis": hyp_id, "outcome": choice},
                        "evidence": ev_text,
                        "hypothesis_id": hyp_id,
                        "hypothesis_text": hypothesis_text,
                    })
                    chunk_idx += 1
    return chunks

structured_chunks = extract_structured_chunks(train_data)
print(f"Extracted {len(structured_chunks)} structured chunks")

client = chromadb.PersistentClient(path=VECTOR_DB_DIR)
if "m2_training_rag" in [c.name for c in client.list_collections()]:
    client.delete_collection("m2_training_rag")
collection = client.create_collection(name="m2_training_rag")

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

if structured_chunks:
    docs = [c["text"] for c in structured_chunks]
    metas = [c["metadata"] for c in structured_chunks]
    ids = [c["id"] for c in structured_chunks]
    print("Embedding...")
    embeddings = embed_model.encode(docs, show_progress_bar=True).tolist()

    BATCH = 5000  # Stay under Chroma's 5461 cap
    for i in range(0, len(docs), BATCH):
        collection.add(
            documents=docs[i:i+BATCH],
            embeddings=embeddings[i:i+BATCH],
            ids=ids[i:i+BATCH],
            metadatas=metas[i:i+BATCH],
        )
        print(f"  inserted batch {i//BATCH + 1}")
    print(f"Vector index persisted to {VECTOR_DB_DIR}")
else:
    print("No structured chunks; skipping vector index build.")

Loaded 423 training documents
Extracted 8341 structured chunks


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding...


Batches:   0%|          | 0/261 [00:00<?, ?it/s]

  inserted batch 1
  inserted batch 2
Vector index persisted to /kaggle/working/vector_db


## 2. Graph RAG Index

Nodes: `Clause`, `Hypothesis`. Edges: directed `Clause -[Entailment|Contradiction]-> Hypothesis`.

In [4]:
import pickle
import networkx as nx

G = nx.DiGraph()
for c in structured_chunks:
    clause_node = f"Clause_{c['metadata']['source_doc']}_{c['id']}"
    rule_node = f"Rule_{c['hypothesis_id']}"
    G.add_node(clause_node, type="Clause", text=c["evidence"], source_doc=c["metadata"]["source_doc"])
    if not G.has_node(rule_node):
        G.add_node(rule_node, type="Hypothesis", text=c["hypothesis_text"])
    G.add_edge(clause_node, rule_node, relation=c["metadata"]["outcome"])

os.makedirs(GRAPH_DB_DIR, exist_ok=True)
with open(GRAPH_PKL, "wb") as f:
    pickle.dump(G, f)
print(f"Graph: {G.number_of_nodes()} nodes / {G.number_of_edges()} edges → {GRAPH_PKL}")

Graph: 8358 nodes / 8341 edges → /kaggle/working/graph_db/contractnli_graph.pkl


## 3. Retrievers

In [5]:
from typing import List, Dict, Any
from sentence_transformers import util as st_util

class VectorRetriever:
    """Top-k semantic retrieval over the persisted ContractNLI training-split corpus."""
    def __init__(self, index_path: str, collection_name: str = "m2_training_rag"):
        self.client = chromadb.PersistentClient(path=index_path)
        self.collection = self.client.get_collection(name=collection_name)
        self.embed_model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")

    def retrieve(self, query: str, top_k: int = 5) -> List[Dict[str, Any]]:
        q_emb = self.embed_model.encode([query]).tolist()
        results = self.collection.query(query_embeddings=q_emb, n_results=top_k)
        out = []
        if results and results["documents"]:
            for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
                out.append({"source": "vector_db", "text": doc, "metadata": meta})
        return out


class GraphRetriever:
    """Semantic entry into Hypothesis nodes, then traverse predecessor Clauses."""
    def __init__(self, graph_path: str, max_clauses_per_rule: int = 20):
        with open(graph_path, "rb") as f:
            self.G = pickle.load(f)
        self.max_clauses_per_rule = max_clauses_per_rule
        self.embed_model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
        self.rule_nodes = [n for n, a in self.G.nodes(data=True) if a.get("type") == "Hypothesis"]
        self.rule_texts = [self.G.nodes[n].get("text", "") for n in self.rule_nodes]
        self.rule_embeddings = self.embed_model.encode(self.rule_texts) if self.rule_texts else None

    def retrieve(self, query: str, top_k_rules: int = 2) -> List[Dict[str, Any]]:
        if not self.rule_nodes:
            return []
        q_emb = self.embed_model.encode(query)
        sims = st_util.cos_sim(q_emb, self.rule_embeddings)[0]
        top_idx = sims.topk(k=min(top_k_rules, len(self.rule_nodes))).indices.tolist()
        out = []
        for idx in top_idx:
            rule_node = self.rule_nodes[idx]
            rule_text = self.rule_texts[idx]
            clauses = list(self.G.predecessors(rule_node))[:self.max_clauses_per_rule]
            for clause_node in clauses:
                clause_data = self.G.nodes[clause_node]
                relation = self.G.get_edge_data(clause_node, rule_node).get("relation", "Unknown")
                chunk_text = (
                    f"Graph Traversal -> Rule: '{rule_text}'\n"
                    f"Historical Outcome: {relation}\n"
                    f"Supporting Clause: '{clause_data.get('text', '')}'"
                )
                out.append({
                    "source": "graph_db",
                    "text": chunk_text,
                    "metadata": {"rule": rule_node, "clause": clause_node, "relation": relation},
                })
        return out


# Smoke test both retrievers (build-time check; the orchestrator will use the configured one).
_test_q = "Does this agreement survive after termination?"
print("-- VectorRetriever --")
_vr = VectorRetriever(VECTOR_DB_DIR)
for r in _vr.retrieve(_test_q, top_k=2):
    print(r["text"][:200], "...")
print("-- GraphRetriever --")
_gr = GraphRetriever(GRAPH_PKL)
for r in _gr.retrieve(_test_q, top_k_rules=1)[:2]:
    print(r["text"][:200], "...")

-- VectorRetriever --


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Hypothesis: 'Some obligations of Agreement may survive termination of Agreement.'
Historical Outcome: Entailment
Supporting Clause: 'This Agreement may be terminated by either party at any time upon t ...
Hypothesis: 'Some obligations of Agreement may survive termination of Agreement.'
Historical Outcome: Entailment
Supporting Clause: '6. This Agreement shall enter into effect upon its execution and sh ...
-- GraphRetriever --


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Graph Traversal -> Rule: 'Some obligations of Agreement may survive termination of Agreement.'
Historical Outcome: Entailment
Supporting Clause: 'Either Party may terminate the working relationship co ...
Graph Traversal -> Rule: 'Some obligations of Agreement may survive termination of Agreement.'
Historical Outcome: Entailment
Supporting Clause: 'Notwithstanding the termination of this Agreement, any ...


## 4. Conversational Agent — Qwen2.5-3B-Instruct + MS1 LoRA Adapter (PEFT)

Loads the 4-bit quantized base model, attaches the adapter from `model-weights/final_adapter/` (PEFT `r=16, alpha=32`, target modules `q/k/v/o/gate/up/down_proj`), and uses the adapter folder's tokenizer + `chat_template.jinja`.

In [6]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

class ConversationalAgent:
    """Stateful conversation layer; preserves history across follow-up turns.

    Grounding rule (per MS2 spec): external RAG context informs reasoning ONLY;
    final answer must be grounded in the target contract text.
    """
    def __init__(self, base_model_id: str = BASE_MODEL_ID, adapter_path: str = None):
        self.history: List[Dict[str, str]] = []
        bnb = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        tokenizer_src = adapter_path if adapter_path else base_model_id
        print(f"[Agent] Tokenizer ← {tokenizer_src}")
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_src, trust_remote_code=True)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        print(f"[Agent] Base model ← {base_model_id} (4-bit nf4)")
        base = AutoModelForCausalLM.from_pretrained(
            base_model_id,
            quantization_config=bnb,
            device_map="auto",
            torch_dtype=torch.float16,
            trust_remote_code=True,
        )

        if adapter_path:
            print(f"[Agent] Attaching PEFT adapter ← {adapter_path}")
            self.model = PeftModel.from_pretrained(base, adapter_path)
        else:
            print("[Agent] No adapter found — running base model only")
            self.model = base
        self.model.eval()

    @torch.inference_mode()
    def generate_response(
        self,
        user_prompt: str,
        contract_text: str,
        context: str,
        max_new_tokens: int = 384,
    ) -> str:
        system_message = (
            "You are an expert legal aide reviewing NDAs. "
            "Use the provided External Context ONLY to understand how similar clauses "
            "were interpreted in the past. Always ground your final decision strictly "
            "on the provided Target Contract Text."
        )
        messages = [{"role": "system", "content": system_message}]

        if not self.history:
            snippet = contract_text if len(contract_text) <= 8000 else contract_text[:8000] + "\n[... truncated ...]"
            first_user = (
                f"### Target Contract:\n{snippet}\n\n"
                f"### Historical Precedents (External RAG Context):\n{context}\n\n"
                f"### User Question:\n{user_prompt}"
            )
            self.history.append({"role": "user", "content": first_user})
        else:
            self.history.append({"role": "user", "content": user_prompt})

        messages.extend(self.history)
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096)
        inputs = {k: v.to(self.model.device) for k, v in inputs.items()}

        out = self.model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            top_p=0.9,
            do_sample=True,
            pad_token_id=self.tokenizer.pad_token_id,
        )
        gen_ids = out[0][inputs["input_ids"].shape[1]:]
        response = self.tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
        self.history.append({"role": "assistant", "content": response})
        return response

    def get_history(self) -> List[Dict[str, str]]:
        return list(self.history)

## 5. Orchestrator + Multi-Turn Demo

Loads an eval-split contract, routes through `RETRIEVER_MODE`, runs an initial turn followed by a follow-up turn that exercises preserved history.

In [7]:
import hashlib
from datetime import datetime, timezone

with open(EVAL_PATH, "r", encoding="utf-8") as f:
    eval_data = json.load(f)

# Pick the first eval contract by default; swap by id for a specific document.
eval_doc = eval_data["documents"][0]
contract_id = str(eval_doc.get("id", "unknown"))
contract_text = eval_doc["text"]
contract_hash = hashlib.sha256(contract_text.encode("utf-8")).hexdigest()
print(f"Eval contract: id={contract_id}  len={len(contract_text)} chars")

if RETRIEVER_MODE == "vector":
    retriever = VectorRetriever(VECTOR_DB_DIR)
else:
    retriever = GraphRetriever(GRAPH_PKL)

agent = ConversationalAgent(base_model_id=BASE_MODEL_ID, adapter_path=ADAPTER_PATH)

conversation_log = []

def turn(user_prompt: str, retrieve_external: bool = True):
    ctxs = retriever.retrieve(user_prompt) if retrieve_external else []
    flat = "\n\n".join(c["text"] for c in ctxs)
    reply = agent.generate_response(user_prompt=user_prompt, contract_text=contract_text, context=flat)
    conversation_log.append({
        "turn_id": len(conversation_log) + 1,
        "role": "user",
        "content": user_prompt,
        "retrieved_context": [c["text"] for c in ctxs],
    })
    conversation_log.append({
        "turn_id": len(conversation_log) + 1,
        "role": "assistant",
        "content": reply,
    })
    print(f"\n--- Turn {len(conversation_log)//2} ---")
    print(f"User: {user_prompt}")
    print(f"AI:   {reply}")
    return reply

_ = turn("Does this contract permit me to disclose information to my employees?")
_ = turn("Based on your previous answer, what should I clarify before signing?", retrieve_external=False)

Eval contract: id=1  len=16632 chars


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Agent] Tokenizer ← /kaggle/input/datasets/yousefelbrolosy/final-adapter-m2/final_adapter
[Agent] Base model ← Qwen/Qwen2.5-3B-Instruct (4-bit nf4)


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[Agent] Attaching PEFT adapter ← /kaggle/input/datasets/yousefelbrolosy/final-adapter-m2/final_adapter

--- Turn 1 ---
User: Does this contract permit me to disclose information to my employees?
AI:   Yes, the contract does permit the disclosure of some Confidential Information to some of the Receiving Party's employees. The specific language that allows this is:

"2. Recipient shall neither use Discloser’s Confidential Information nor circulate it within its own organization, except to the extent necessary for negotiations, discussions and consultations with personnel or authorized representatives of Discloser and then only to those who have been advised that they are subject to the terms of this Agreement and have signed a non-disclosure or other equivalent agreement that protects the Confidential Information."
However, these employees must be employees of the Receiving Party, not of any third party.

--- Turn 2 ---
User: Based on your previous answer, what should I clarify before si

## 6. Runtrace Output

Conforms to `assets/runtrace_ms2.schema.json`. `hypothesis_traces` and `metrics` are placeholders for the MS3 17-hypothesis fan-out.

In [ ]:
from jsonschema import Draft202012Validator

RUNTRACE_SCHEMA_PATH = os.path.join(WORK_DIR, "assets", "runtrace_ms2.schema.json")
PLAYBOOK_SCHEMA_PATH = os.path.join(WORK_DIR, "assets", "playbook_ms2.schema.json")

with open(RUNTRACE_SCHEMA_PATH, "r", encoding="utf-8") as f:
    runtrace_schema = json.load(f)

Draft202012Validator.check_schema(runtrace_schema)

runtrace = {
    "schema_version": "2.0-ms2",
    "run": {
        "run_id": f"ms2-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}",
        "started_at": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
        "framework": "multi_agent_system",
        "status": "success",
    },
    "retrieval_strategy": {
        "mode": RETRIEVER_MODE,
        "knowledge_base_source": "ContractNLI-train",
        "selector": "RETRIEVER_MODE environment variable or notebook default",
    },
    "conversation_history": conversation_log,
    "contract": {
        "contract_id": contract_id,
        "text": contract_text[:2000],
        "hash_sha256": contract_hash,
        "split": "ContractNLI evaluation split",
    },
    "playbook": {
        "schema_version": "2.0-ms2",
        "schema_file": "assets/playbook_ms2.schema.json",
        "status": "schema_defined_ms2_full_playbook_instance_pending",
    },
    "hypothesis_traces": [],
    "metrics": {},
    "run_validations": [
        {
            "name": "runtrace_schema_loaded",
            "status": "pass",
            "details": "Loaded and checked assets/runtrace_ms2.schema.json with Draft202012Validator.",
        }
    ],
}

Draft202012Validator(runtrace_schema).validate(runtrace)
runtrace["run_validations"].append({
    "name": "runtrace_schema_validation",
    "status": "pass",
    "details": "Generated runtrace validates against assets/runtrace_ms2.schema.json.",
})

with open(RUNTRACE_OUT, "w", encoding="utf-8") as f:
    json.dump(runtrace, f, indent=2)
print(f"Runtrace schema -> {RUNTRACE_SCHEMA_PATH}")
print(f"Playbook schema  -> {PLAYBOOK_SCHEMA_PATH}")
print(f"Runtrace validated and written -> {RUNTRACE_OUT}")
